# SD3.5 mask-free EDIT — generate → YOLO-segment → paste into original

Inference-only. Implements the 2026-06-27 pipeline:

1. the **mask-free** model (LoRA + `input_proj.pt`) generates a FULL image that
   adds a person fitting the scene's **vibe** — no mask, no background preserve.
2. **YOLOv8-seg** segments the generated person out.
3. that person is **pasted onto the ORIGINAL background** at the same coords,
   feathered + colour-matched so it isn't a sticker.

Net effect: background is preserved byte-exact OUTSIDE the person (composite,
not hard-restore), while the person comes from a model free to make it look
natural in-scene. The harmonisation cost (person lit by the *generated* scene)
is handled by the colour-match in `segment_paste.composite_persons`.

Needs GPU + SD3.5 access + a trained mask-free adapter (mount your run as a dataset).

## 1. Setup

In [ ]:
import subprocess, sys, os
from pathlib import Path
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
sys.path.insert(0, str(REPO))
print('repo at', subprocess.run(['git','-C',str(REPO),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())
subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps',
                'transformers==4.46.3','tokenizers==0.20.3','huggingface_hub==0.25.2'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
                'diffusers==0.31.0','accelerate==0.34.2','peft==0.13.2',
                'safetensors>=0.4.3','sentencepiece','protobuf','pillow>=10','numpy'], check=True)
# ultralytics (YOLOv8-seg) does the person segmentation; --no-deps keeps the pins above.
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','ultralytics'], check=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import transformers, diffusers, torch
assert transformers.__version__ == '4.46.3'
from transformers.utils import FLAX_WEIGHTS_NAME
assert torch.cuda.is_available(); print('OK on', torch.cuda.get_device_name(0))

## 2. SD3.5 access + locate the trained mask-free adapter

In [ ]:
from pathlib import Path
_local = Path('/kaggle/input/stable-diffusion-3-5-medium')
HF_TOKEN = None
if _local.exists():
    SD35_MODEL = str(_local)
else:
    SD35_MODEL = 'stabilityai/stable-diffusion-3.5-medium'
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        import os; HF_TOKEN = os.environ.get('HF_TOKEN')
    assert HF_TOKEN, 'Need HF_TOKEN or mounted SD3.5'
    from huggingface_hub import login; login(token=HF_TOKEN)

ADAPTER_RUN = None
if ADAPTER_RUN is None:
    cands = [p.parent for p in Path('/kaggle/input').rglob('input_proj.pt')]
    cands += [p.parent for p in Path('/kaggle/working/vin_lora/models').rglob('input_proj.pt')] if Path('/kaggle/working/vin_lora/models').exists() else []
    ADAPTER_RUN = cands[0].parent if cands else None
else:
    ADAPTER_RUN = Path(ADAPTER_RUN)
assert ADAPTER_RUN is not None and (ADAPTER_RUN/'adapter'/'input_proj.pt').exists(), (
    'No mask-free adapter found. Mount the trained run as a dataset (folder with '
    'training_provenance.json + adapter/{pytorch_lora_weights.safetensors,input_proj.pt}) '
    'or set ADAPTER_RUN explicitly.')
print('adapter run:', ADAPTER_RUN)

## 3. Load the mask-free runner + the YOLOv8-seg segmenter (once)

In [ ]:
from LoRA.inference.sd35_maskfree_runner import load_maskfree_runner_from_run
from LoRA.inference.person_detector import load_person_segmenter
runner, prov = load_maskfree_runner_from_run(ADAPTER_RUN, base_model_id=SD35_MODEL, hf_token=HF_TOKEN)
assert prov.get('requires_input_proj'), 'not a mask-free adapter (missing input_proj flag)'
# YOLOv8-seg on CPU keeps the T4 free for SD3.5; person masks on 512px are cheap.
segmenter = load_person_segmenter('yolov8n-seg.pt', device='cpu')
print('runner + segmenter ready | mask_free =', prov.get('mask_free'))

## 4. Background images (from a dataset in sources.yaml) + settings

`BG_SOURCE` = `citypersons` | `mot17_02` | `human_detection`. Pulls real images
straight from the mounted dataset; falls back to `/kaggle/working/my_backgrounds`.

In [ ]:
from pathlib import Path
from LoRA.data.list_images import list_source_images

BG_SOURCE = 'citypersons'   # 'citypersons' | 'mot17_02' | 'human_detection'
N_BG = 6                    # how many images to process
STEPS = 30
S_IMAGE = 1.5               # source-image adherence (CFG) — higher = generated scene stays closer to source
S_TEXT = 7.5                # instruction adherence (CFG)
FEATHER_PX = 3              # seam softness when pasting the person
COLOR_MATCH = 0.5           # 0..1 colour transfer toward the original (anti-sticker)
POISSON = False             # True = OpenCV seamlessClone for the largest person
# Plain natural-language instruction — the mask-free flow trains on raw PIPE
# instructions and DROPPED the <vin_ped> trigger token, so do NOT add it here.
ADD_PROMPT = 'add a person walking'
SOURCES_YAML = Path('/kaggle/working/VIN/LoRA/configs/sources.yaml')

try:
    bg_paths = list_source_images(BG_SOURCE, limit=N_BG, sources_path=SOURCES_YAML)
    print(f'{len(bg_paths)} images from dataset source "{BG_SOURCE}"')
except FileNotFoundError as e:
    print('dataset not mounted:', e)
    BG_DIR = Path('/kaggle/working/my_backgrounds'); BG_DIR.mkdir(parents=True, exist_ok=True)
    bg_paths = sorted([p for p in BG_DIR.glob('*') if p.suffix.lower() in ('.png','.jpg','.jpeg')])[:N_BG]
    print(f'{len(bg_paths)} images from {BG_DIR}')
assert bg_paths, f'No images for "{BG_SOURCE}" (mount via Add Data) or in the fallback folder.'

## 5. Generate → segment → paste, per image

In [ ]:
import time
from PIL import Image
from LoRA.inference.segment_paste import generate_and_paste
OUT = Path('/kaggle/working/maskfree_segpaste'); (OUT/'images').mkdir(parents=True, exist_ok=True)
runner.precompute_embeds([ADD_PROMPT])
results = []
for i, bp in enumerate(bg_paths):
    orig = Image.open(bp).convert('RGB')
    te = time.time()
    composite, generated, info = generate_and_paste(
        runner, orig, ADD_PROMPT, segmenter,
        seed=42, num_inference_steps=STEPS, s_image=S_IMAGE, s_text=S_TEXT,
        feather_px=FEATHER_PX, color_match=COLOR_MATCH, poisson=POISSON)
    composite.save(OUT/'images'/f'{bp.stem}_segpaste.png')
    results.append((bp.stem, orig, generated, composite, info))
    print(f'[{i+1}/{len(bg_paths)}] {bp.name}  {time.time()-te:.1f}s  pasted={info["pasted"]} confs={info.get("confs")}', flush=True)
print('outputs ->', OUT)

## 6. Show: original | generated (full) | composite (person pasted onto original)

In [ ]:
from PIL import Image
from IPython.display import display
for name, orig, generated, composite, info in results:
    cells = [orig.resize((256,256)), generated.resize((256,256)), composite.resize((256,256))]
    strip = Image.new('RGB', (768, 256))
    for j, im in enumerate(cells):
        strip.paste(im, (256*j, 0))
    print(f'{name}  (original | generated | composite)  pasted={info["pasted"]}')
    display(strip)